In [2]:
import json
import os
import pandas as pd
from typing import List, Set

# ==========================================
# 1. Core configuration section (please modify paths to match your environment)
# ==========================================

# [Input] Raw NII image folder path
NII_ROOT_DIR = "/home/baidu/scratch/File_transfer/abdomen_disease_classify/imagesTr"

# [Input] Split file path (split.json)
SPLIT_JSON_PATH = "/home/baidu/scratch/File_transfer/abdomen_disease_classify/cls_labels/splits_final_adrenal_hyperplasia.json"  # <--- Please change to the real path

# [Input] Preprocessed cache data root directory (used to generate the csv)
CACHE_ROOT_DIR = "/home/baidu/scratch/Pillar_Eval/RAVE_preprocessed/abdomen_disease_classify" # <--- Please change to the real path

# [Output] Dataset configuration directory (location for .json and .csv files)
DATASET_CONFIG_DIR = "/home/baidu/scratch/Pillar_Eval/rate-evals"
CACHE_MANIFEST_DIR = "/home/baidu/scratch/Pillar_Eval/rate-evals"

# ==========================================
# 2. Helper functions
# ==========================================

def generate_jsonl_lines(sample_names: List[str], nii_root: str) -> List[str]:
    lines = []
    for name in sample_names:
        # Even though the cache takes priority, we still need to provide the NII path as a fallback
        nii_path = os.path.join(nii_root, f"{name}.nii.gz")
        sample_obj = {
            "sample_name": name, 
            "nii_path": nii_path, 
            "report_metadata": {}
        }
        lines.append(json.dumps(sample_obj, ensure_ascii=False))
    return lines

def save_file(path: str, content: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"✅ Generated file: {path}")

def match_folder_to_id(folder_name: str, valid_ids: Set[str]) -> str:
    """
    Try to map a cache folder name (e.g. amos_0004.1.0) back to a standard ID (e.g. amos_0004)
    """
    # 1. Try exact match
    if folder_name in valid_ids:
        return folder_name
    
    # 2. Try match after stripping suffix (assuming suffix starts with .)
    # Example: amos_0004.1.0 -> amos_0004
    base_name = folder_name.split('.')[0]
    if base_name in valid_ids:
        return base_name
    
    # 3. If the ID itself contains . (uncommon), try a prefix match
    # Iterate over all valid IDs and see whether the folder name begins with it
    for vid in valid_ids:
        if folder_name.startswith(vid):
            # Ensure a clean match boundary (e.g. to prevent case_1 matching case_10)
            remainder = folder_name[len(vid):]
            if remainder == "" or remainder.startswith(".") or remainder.startswith("_"):
                return vid
                
    return None

def process_all():
    print("🚀 Starting processing...")
    
    # --- Step 1: Read the split file to get standard IDs ---
    if not os.path.exists(SPLIT_JSON_PATH):
        print(f"❌ Split file not found: {SPLIT_JSON_PATH}")
        return

    with open(SPLIT_JSON_PATH, "r", encoding="utf-8") as f:
        split_data = json.load(f)
    
    # Collect all valid IDs (used to validate cache folders)
    train_ids = split_data.get("train", [])
    val_ids = split_data.get("val", [])
    all_valid_ids = set(train_ids + val_ids)
    
    print(f"📊 Total standard IDs: {len(all_valid_ids)} (Train: {len(train_ids)}, Val: {len(val_ids)})")

    # --- Step 2: Generate JSONL configuration files ---
    print("\n--- Generating Dataset JSONL ---")
    
    # train.json
    save_file(
        os.path.join(DATASET_CONFIG_DIR, "train.json"), 
        "\n".join(generate_jsonl_lines(train_ids, NII_ROOT_DIR)) + "\n"
    )
    
    # valid.json & test.json (both use the val data)
    val_content = "\n".join(generate_jsonl_lines(val_ids, NII_ROOT_DIR)) + "\n"
    save_file(os.path.join(DATASET_CONFIG_DIR, "valid.json"), val_content)
    save_file(os.path.join(DATASET_CONFIG_DIR, "test.json"), val_content)

    # --- Step 3: Generate the corrected Manifest CSV ---
    print("\n--- Generating Cache Manifest CSV (with ID fix) ---")
    
    if not os.path.exists(CACHE_ROOT_DIR):
        print(f"❌ Cache directory does not exist: {CACHE_ROOT_DIR}")
        return

    manifest_data = []
    matched_count = 0
    unmatched_count = 0
    
    # Scan the cache directory
    cache_folders = sorted([d for d in os.listdir(CACHE_ROOT_DIR) if os.path.isdir(os.path.join(CACHE_ROOT_DIR, d))])
    
    for folder_name in cache_folders:
        folder_path = os.path.join(CACHE_ROOT_DIR, folder_name)
        
        # Try to match
        matched_id = match_folder_to_id(folder_name, all_valid_ids)
        
        if matched_id:
            # Successfully matched!
            # In the CSV, sample_name stores the standard ID (amos_0004)
            # image_cache_path stores the actual folder path (.../amos_0004.1.0)
            manifest_data.append({
                "sample_name": matched_id,
                "image_cache_path": os.path.abspath(folder_path)
            })
            matched_count += 1
        else:
            # Could not match - log a warning
            unmatched_count += 1
            # print(f"⚠️ Skipping unrecognized folder: {folder_name}")

    # Save the CSV
    output_csv_path = os.path.join(CACHE_MANIFEST_DIR, "manifest.csv")
    os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
    
    df = pd.DataFrame(manifest_data)
    if not df.empty:
        df = df.sort_values("sample_name")
        df.to_csv(output_csv_path, index=False)
        print(f"✅ CSV generated: {output_csv_path}")
        print(f"   - Successfully matched samples: {matched_count}")
        if unmatched_count > 0:
            print(f"   - Unmatched folders: {unmatched_count} (likely extra data not in split)")
        
        # Preview
        print("\nCSV content preview (first 3 rows):")
        print(df.head(3))
    else:
        print("❌ Generation failed: no valid samples matched.")

# ==========================================
# Execute
# ==========================================
if __name__ == "__main__":
    process_all()

🚀 Starting processing...
📊 Total standard IDs: 1005 (Train: 935, Val: 70)

--- Generating Dataset JSONL ---
✅ Generated file: /home/baidu/scratch/Pillar_Eval/rate-evals/train.json
✅ Generated file: /home/baidu/scratch/Pillar_Eval/rate-evals/valid.json
✅ Generated file: /home/baidu/scratch/Pillar_Eval/rate-evals/test.json

--- Generating Cache Manifest CSV (with ID fix) ---
✅ CSV generated: /home/baidu/scratch/Pillar_Eval/rate-evals/manifest.csv
   - Successfully matched samples: 1005
   - Unmatched folders: 460 (likely extra data not in split)

CSV content preview (first 3 rows):
  sample_name                                   image_cache_path
0   amos_0004  /home/baidu/scratch/Pillar_Eval/RAVE_preproces...
1   amos_0005  /home/baidu/scratch/Pillar_Eval/RAVE_preproces...
2   amos_0006  /home/baidu/scratch/Pillar_Eval/RAVE_preproces...
